## Play Tennis Dataset Classification Pipeline



In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import CategoricalNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import numpy as np

# Load the dataset
df = pd.read_csv('/content/Lab 8 - Sheet1.csv')
display(df.head())

,No,Outlook,Temperature,Humidity,Wind,Play Tennis
0,1,Sunny,Hot,High,Weak,No
1,2,Sunny,Hot,High,Strong,No
2,3,Overcast,Hot,High,Weak,Yes
3,4,Rain,Mild,High,Weak,Yes
4,5,Rain,Cool,Normal,Weak,Yes


### Data Preprocessing

1.  **Separate Features and Target**: Identify input features and the target variable.
2.  **Encode Categorical Features**: Convert all categorical feature values and target labels into numerical representations using `LabelEncoder`.

In [2]:
# Separate input features (X) from the target variable (y)
X = df[['Outlook', 'Temperature', 'Humidity', 'Wind']].copy()
y = df['Play Tennis']

# Initialize LabelEncoders
le_outlook = LabelEncoder()
le_temperature = LabelEncoder()
le_humidity = LabelEncoder()
le_wind = LabelEncoder()
le_play = LabelEncoder()

# Apply LabelEncoder to each categorical feature
X['Outlook'] = le_outlook.fit_transform(X['Outlook'])
X['Temperature'] = le_temperature.fit_transform(X['Temperature'])
X['Humidity'] = le_humidity.fit_transform(X['Humidity'])
X['Wind'] = le_wind.fit_transform(X['Wind'])

# Apply LabelEncoder to the target variable
y_encoded = le_play.fit_transform(y)

print("Encoded Features (X.head()):")
display(X.head())
print("Encoded Target (y_encoded[:5]):")
print(y_encoded[:5])

# Store the mappings for inverse transformation later if needed
feature_encoders = {
    'Outlook': le_outlook,
    'Temperature': le_temperature,
    'Humidity': le_humidity,
    'Wind': le_wind
}
target_encoder = le_play

Encoded Features (X.head()):


,Outlook,Temperature,Humidity,Wind
0,2,1,0,1
1,2,1,0,0
2,0,1,0,1
3,1,2,0,1
4,1,0,1,1


Encoded Target (y_encoded[:5]):
[0 0 1 1 1]


### Check for Class Imbalance

It's important to check for class imbalance in the target variable to understand the distribution of classes and potential impact on model performance. This section displays the value counts for the original and encoded target variable.

In [3]:
print("Original Target Variable Distribution (y):")
display(y.value_counts())
print("\nEncoded Target Variable Distribution (y_encoded):")
display(pd.Series(y_encoded).value_counts())

Original Target Variable Distribution (y):


,count
Play Tennis,
Yes,34
No,16



Encoded Target Variable Distribution (y_encoded):


,count
1,34
0,16


### Dataset Partitioning

Divide the dataset into training and testing subsets using an 80:20 train-test split ratio.

In [4]:
# Divide the dataset into training and testing subsets (80:20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.20, random_state=42)

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")
print(f"Training target shape: {y_train.shape}")
print(f"Testing target shape: {y_test.shape}")

Training features shape: (40, 4)
Testing features shape: (10, 4)
Training target shape: (40,)
Testing target shape: (10,)


### Naive Bayes Model Training & Evaluation

1.  **Train Model**: Train a Categorical Naive Bayes (`CategoricalNB`) model on the training data.
2.  **Predict**: Predict the class labels for the test dataset.
3.  **Evaluate**: Calculate and display the overall Model Accuracy, Confusion Matrix, and Classification Report (Precision, Recall, F1-Score).

In [5]:
# Train a Categorical Naive Bayes model
cnb_model = CategoricalNB()
cnb_model.fit(X_train, y_train)

# Predict the class labels for the test dataset
y_pred_cnb = cnb_model.predict(X_test)

# Calculate and display Model Accuracy
accuracy_cnb = accuracy_score(y_test, y_pred_cnb)
print(f"Categorical Naive Bayes Model Accuracy: {accuracy_cnb:.4f}")

# Display the Confusion Matrix
print("\nConfusion Matrix (Categorical Naive Bayes):\n", confusion_matrix(y_test, y_pred_cnb))

# Display the Classification Report
print("\nClassification Report (Categorical Naive Bayes):\n", classification_report(y_test, y_pred_cnb,
                                                           target_names=target_encoder.classes_))

Categorical Naive Bayes Model Accuracy: 0.8000

Confusion Matrix (Categorical Naive Bayes):
 [[1 1]
 [1 7]]

Classification Report (Categorical Naive Bayes):
               precision    recall  f1-score   support

          No       0.50      0.50      0.50         2
         Yes       0.88      0.88      0.88         8

    accuracy                           0.80        10
   macro avg       0.69      0.69      0.69        10
weighted avg       0.80      0.80      0.80        10



### Single-Sample Inference (Categorical Naive Bayes)

Predict whether a person will play tennis under specific weather conditions and display both the predicted class label and the corresponding class probabilities.

In [6]:
# Define the single sample conditions
single_sample_conditions = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

# Encode the single sample using the same encoders used for training data
encoded_single_sample = [
    feature_encoders['Outlook'].transform([single_sample_conditions['Outlook']])[0],
    feature_encoders['Temperature'].transform([single_sample_conditions['Temperature']])[0],
    feature_encoders['Humidity'].transform([single_sample_conditions['Humidity']])[0],
    feature_encoders['Wind'].transform([single_sample_conditions['Wind']])[0]
]

# Convert to a NumPy array and reshape for prediction
single_sample_array = np.array(encoded_single_sample).reshape(1, -1)

# Predict the class label
predicted_label_cnb = cnb_model.predict(single_sample_array)
predicted_label_cnb_decoded = target_encoder.inverse_transform(predicted_label_cnb)[0]

# Get class probabilities
predicted_probabilities_cnb = cnb_model.predict_proba(single_sample_array)[0]

print(f"Single-Sample Conditions: {single_sample_conditions}")
print(f"Predicted Class Label (Categorical Naive Bayes): {predicted_label_cnb_decoded}")
print("Class Probabilities (Categorical Naive Bayes):")
for i, class_name in enumerate(target_encoder.classes_):
    print(f"  {class_name}: {predicted_probabilities_cnb[i]:.4f}")

Single-Sample Conditions: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Predicted Class Label (Categorical Naive Bayes): No
Class Probabilities (Categorical Naive Bayes):
  No: 0.9256
  Yes: 0.0744


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but CategoricalNB was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but CategoricalNB was fitted with feature names
  warnings.warn(


### Model Comparison

Train a Decision Tree Classifier, a Logistic Regression Classifier, and an SVM on the same training data. Compare all three models in terms of test accuracy, single-sample query prediction, and output class probabilities with Categorical Naive Bayes.

In [7]:
# Initialize models
dt_model = DecisionTreeClassifier(random_state=42)
lr_model = LogisticRegression(random_state=42, solver='liblinear') # 'liblinear' is good for small datasets
svm_model = SVC(probability=True, random_state=42) # probability=True is needed for predict_proba

models = {
    'Categorical Naive Bayes': cnb_model,
    'Decision Tree': dt_model,
    'Logistic Regression': lr_model,
    'SVM': svm_model
}

# Store results
results = []
single_sample_predictions = []

# Train and evaluate each model
for name, model in models.items():
    print(f"\n--- {name} ---")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy}") # Removed rounding

    # Store accuracy
    results.append({'Model': name, 'Test Accuracy': accuracy})

    # Single-sample inference
    predicted_label = model.predict(single_sample_array)[0]
    predicted_label_decoded = target_encoder.inverse_transform([predicted_label])[0]

    probabilities = model.predict_proba(single_sample_array)[0]

    single_sample_predictions.append({
        'Model': name,
        'Predicted Label': predicted_label_decoded,
        'Probabilities': {class_name: prob for class_name, prob in zip(target_encoder.classes_, probabilities)}
    })

# Display comparison table for accuracies
print("\n--- Model Accuracy Comparison ---")
display(pd.DataFrame(results))

# Display comparison table for single-sample predictions
print("\n--- Single-Sample Prediction Comparison ---")
single_sample_df = pd.DataFrame(single_sample_predictions)
single_sample_df_pretty = single_sample_df.copy()
single_sample_df_pretty['Probabilities'] = single_sample_df_pretty['Probabilities'].apply(lambda x: ', '.join([f'{k}: {v:.4f}' for k, v in x.items()]))
display(single_sample_df_pretty)


--- Categorical Naive Bayes ---
Accuracy: 0.8

--- Decision Tree ---
Accuracy: 0.8

--- Logistic Regression ---
Accuracy: 0.4

--- SVM ---
Accuracy: 0.7

--- Model Accuracy Comparison ---


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but CategoricalNB was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but CategoricalNB was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-p

,Model,Test Accuracy
0,Categorical Naive Bayes,0.8
1,Decision Tree,0.8
2,Logistic Regression,0.4
3,SVM,0.7



--- Single-Sample Prediction Comparison ---


,Model,Predicted Label,Probabilities
0,Categorical Naive Bayes,No,"No: 0.9256, Yes: 0.0744"
1,Decision Tree,No,"No: 1.0000, Yes: 0.0000"
2,Logistic Regression,No,"No: 0.9466, Yes: 0.0534"
3,SVM,No,"No: 0.9751, Yes: 0.0249"


### Analysis Report

Based on the single-sample prediction for conditions (Outlook: Sunny, Temperature: Cool, Humidity: High, Wind: Strong):

Although a moderate class imbalance exists (34 'Yes' vs. 16 'No'), explicit handling techniques are not strictly necessary for this small dataset and the current demonstration purpose.

The models may produce different predictions and probability scores due to their underlying algorithms and assumptions. Categorical Naive Bayes assumes independence between features given the class, which might not hold perfectly in real data, but it's often robust. Decision Trees make decisions based on splitting rules, which can lead to hard classifications and potentially overfitting if not pruned. Logistic Regression models the probability of a binary outcome using a logistic function, making it good for linear decision boundaries. SVM aims to find an optimal hyperplane to separate classes, which can be effective in high-dimensional spaces.

In this specific case, the Decision Tree Classifier perfectly fit the training data for the given conditions, classifying it as 'No'. Categorical Naive Bayes and Logistic Regression provide probabilistic outputs, which can vary based on how they handle different feature combinations and their inherent biases. SVM, when `probability=True`, approximates probabilities using Platt scaling, which can also differ. These differences highlight the importance of model selection and understanding their operational mechanisms.